In [ ]:
import torch

a = [[1,2,3],[4,5,6]]

print(torch.tensor(a,device='mps'))

In [ ]:
print(torch.empty(2,3).to('mps'))

In [ ]:
print(torch.randn(2,3))

In [ ]:
import numpy as np
a = np.array([[1,2,3],[4,5,6]])
print(torch.from_numpy(a))

In [ ]:
a = torch.ones(2,3,4,dtype=torch.float).to('mps')
print(a.shape)
print(a.dtype)
print(a.device)

In [ ]:
a = np.array([[1,2,3],[4,5,6]])
b = torch.tensor([[1,1,1],[2,2,2]])
a = torch.tensor(a)

print(a)
print(b)
print(a + b)
print(a * b)
print(a ** 2)

In [ ]:
a = torch.rand(3,4)
b = torch.rand(4,5)
print(torch.mm(a,b))
print(a@b)

In [ ]:
x = torch.rand(2,3,4)
print(x)
y = x.reshape(2,12)
# print(y)
z = x.transpose(0,2)
print(z)
z = x.permute(2,1,0)
# print(z)

In [ ]:
w = torch.tensor([3.0],requires_grad=True)
b = torch.tensor([2.0],requires_grad=True)
x = torch.tensor([4.0],requires_grad=True)

y = w*x + b
# y.requires_grad_()
target = torch.tensor([10.0])
loss = (y - target) ** 2
print(y)
print(loss)

y.retain_grad()
loss.backward()
# y.backward()
print(y.grad)
print(x.grad)
print(w.grad)
print(b.grad)

In [ ]:
import torch
torch.manual_seed(42)
X = torch.rand(100,1, device='cuda') * 10
true_w = 3.0
true_b = 2.0
Y = true_w * X + true_b + torch.randn(100,1, device='cuda') * 2

w = torch.randn(1, requires_grad=True, device='cuda')
b = torch.zeros(1, requires_grad=True, device='cuda')

learning_rate = 0.01
epochs = 100

for epoch in range(epochs):
    y_pred = w * X + b
    loss = torch.mean((y_pred - Y) ** 2)
    
    loss.backward()
    
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        
        w.grad.zero_()
        b.grad.zero_()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}, w: {w.item():.4f}, b: {b.item():.4f}')

In [ ]:
import torch
torch.manual_seed(42)
X = torch.rand(100,1) * 0.1
true_w1 = 3.0
true_w2 = 4.0
true_b = 2.0
Y = true_w1 * X**2 + true_w2 * X + true_b + torch.randn(100,1) * 2
print(Y[:5])

w1 = torch.tensor([2.0], requires_grad=True)
w2 = torch.tensor([5.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

learning_rate = 0.01
epochs = 100

for epoch in range(epochs):
    y_pred = w1 * X**2 + w2 * X + b
    loss = torch.mean((y_pred - Y) ** 2)
    
    loss.backward()
    
    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad
        b -= learning_rate * b.grad
        
        w1.grad.zero_()
        w2.grad.zero_()
        b.grad.zero_()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}, w1: {w1.item():.4f}, w2: {w2.item():.4f}, b: {b.item():.4f}')

In [ ]:
import torch
from torch import nn
import torch.optim as optim

class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)
    
model = LinearModel()
# print(model)

# for name, param in model.named_parameters():
#     print(name, param.shape, param.requires_grad)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

torch.manual_seed(42)
X = torch.rand(100, 1) * 10
true_w, true_b = 3.0, 2.0
Y = true_w * X + true_b + torch.randn(100, 1) * 2

epochs = 200

for epoch in range(epochs):
    y_pred = model(X)
    loss = criterion(y_pred, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 40 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


[w, b] = model.linear.parameters()
print(f'w = {w.item():.3f}, b = {b.item():.3f}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

X = torch.linspace(-3, 3, 200).unsqueeze(1)

Y = torch.tensor(X ** 2) + 0.1 * torch.randn(X.size())

# print(X[:5])
# print(Y[:5])

class simpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(1,10)
        self.output = nn.Linear(10,1)
        self.activation = nn.Tanh()

    def forward(self,x):
        x = self.hidden(x)
        x = self.activation(x)
        x = self.output(x)
        return x

model = simpleMLP()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 1000

for epoch in range(epochs):
    y_pred = model(X)
    loss = criterion(y_pred, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 200 == 0:
        print(f'Epoch: {epoch+1}, Loss: {loss.item():.4f}')


import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    y_pred = model(X).detach()

plt.scatter(X,Y,label='True data', s=10)
plt.plot(X,y_pred, 'r', lw=2, label="Fitted curve")
plt.legend()
plt.show()


    


In [ ]:
from torch.utils.data import Dataset, IterableDataset, DataLoader
import torch
import torch.optim as optim
from torch import nn

class Mydataset(Dataset):
    def __init__(self,n):
        self.X = torch.linspace(0,7,n).unsqueeze(1)
        self.Y = torch.sin(self.X) + torch.randn(self.X.size()) * 0.1

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.Y[index]
    
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(1,10)
        self.activation = nn.ReLU()
        self.output = nn.Linear(10,1)

    def forward(self,x):
        x = self.hidden(x)
        x = self.activation(x)
        x = self.output(x)
        return x

model = SimpleMLP().to('cuda')
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


mydataset = Mydataset(100)
mydataloader = DataLoader(mydataset, batch_size=50, shuffle=True)

epochs = 1000
for epoch in range(epochs):
    for x_batch,y_batch in mydataloader:
        x_batch = x_batch.to('cuda')
        y_batch = y_batch.to('cuda')
        y_batch_pred = model(x_batch)
        loss = criterion(y_batch_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (epoch+1) % 200 == 0:
            print(f'epoch: {epoch+1}, loss: {loss.item():.4f}')


In [ ]:
from torch.utils.data import Dataset, IterableDataset, DataLoader
import torch
import torch.optim as optim
from torch import nn
from tqdm import tqdm
import time

class Mydata(IterableDataset):
    def __init__(self, n):
        self.X = torch.linspace(0,7,n)
        self.Y = torch.sin(self.X) + torch.randn(self.X.size())*0.1

    def __iter__(self):
        for i in range(len(self.X)):
            yield [self.X[i], self.Y[i]]

n=100
mydata = Mydata(n)
batch_size = 5
mydataloader = DataLoader(mydata, batch_size=batch_size)

# for item in mydataloader:
#     print(item)

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.stack = nn.Sequential(
            nn.Linear(1,5),
            nn.ReLU(),
            nn.Linear(5,3),
            nn.Sigmoid(),
            nn.Linear(3,1),
            nn.Dropout(p=0.1),
        )

    def forward(self, x):
        return self.stack(x)
    
model = SimpleMLP()
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr = 0.01)

# print(model)

epochs = 3
for i in range(epochs):
    print(f'Epoch:{i+1}/{epochs}---------------------------------------')
    pbar = tqdm(total=n)
    for x_batch,y_batch in mydataloader:
        x_batch = x_batch.unsqueeze(1)
        y_batch = y_batch.unsqueeze(1)
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.update(batch_size)
        pbar.set_description(f'loss:{loss:.3f} ')
        time.sleep(0.1)
    pbar.close()

x = torch.linspace(0,7,10)
x = x.unsqueeze(1)
y_real = torch.sin(x)

model.eval()
with torch.no_grad():
    y_pred = model(x)
print(y_real)
print(y_pred)

import matplotlib.pyplot as plt
plt.scatter(x,y_real,label='True data', s=10)
plt.scatter(x,y_pred, label="Fitted curve")
plt.legend()
plt.show()

In [ ]:
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, IterableDataset, DataLoader
import torch
import torch.optim as optim
from torch import nn
import time


# 定义图像预处理：转成 Tensor，并归一化到均值0.5，标准差0.5
transform = transforms.Compose([
    transforms.ToTensor(),                     # 自动将 PIL Image 或 numpy 转为 (C,H,W) 张量，且值域[0,1]
    transforms.Normalize((0.5,), (0.5,))       # 单通道灰度图，均值和标准差各一个值
])

# 下载并加载训练集
train_dataset = torchvision.datasets.MNIST(
    root='./data',          # 数据存放目录
    train=True,
    transform=transform,
    download=True           # 如果没下载过会自动下载
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

# 构建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)  # 测试集不需要打乱

images, labels = next(iter(train_loader))
print(images.shape)   # torch.Size([64, 1, 28, 28])  -> [batch, 通道数, 高度, 宽度]
print(labels.shape)   # torch.Size([64])

# 获取一个批次的数据
dataiter = iter(train_loader)
images, labels = next(dataiter)

# 选一张图（比如第0张）
img = images[6]           # 形状: (1, 28, 28)

# 1. 反归一化： (x * std) + mean
mean = 0.5
std = 0.5
img = img * std + mean    # 变回 [0, 1] 范围

# 2. 转成 (H, W) 的 numpy 数组
img = img.squeeze().numpy()   # 去掉通道维，变成 (28,28)

# 3. 显示灰度图
# plt.imshow(img, cmap='gray')
# plt.title(f'Label: {labels[0].item()}')
# plt.axis('off')
# plt.show()


class MNIST_Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.stack = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),   # 输入784 -> 隐藏128
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),    # 输出10类
        )

    def forward(self, x):
        # x 形状: (batch, 1, 28, 28)
        x = x.view(x.size(0), -1)
        # 交叉熵损失里会包含 softmax，这里直接输出 logits
        return self.stack(x)
    
model = MNIST_Classifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# devic = 'cpu'
model.to(device)

epochs = 5
for epoch in range(epochs):
    model.train()   # 训练模式
    total_loss = 0
    correct = 0
    total = 0
    print(f'Epoch:{epoch+1}/{epochs}------------------------------------')
    pbar = tqdm(total=len(train_dataset))
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # 前向
        outputs = model(images)
        loss = criterion(outputs, labels)

        # 反向
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 统计
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)      # 等价于 argmax
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.update(64)
        pbar.set_description(f'loss:{loss:.4f} ')
    pbar.close()
    epoch_loss = total_loss / total
    epoch_acc = correct / total
    print(f'Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')


model.eval()    # 评估模式（会关闭 Dropout 等）
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_loss /= total
test_acc = correct / total
print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}')

In [41]:
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, IterableDataset, DataLoader
import torch
import torch.optim as optim
from torch import nn
import time

# 定义图像预处理：转成 Tensor，并归一化到均值0.5，标准差0.5
transform = transforms.Compose([
    transforms.ToTensor(),                     # 自动将 PIL Image 或 numpy 转为 (C,H,W) 张量，且值域[0,1]
    transforms.Normalize((0.5,), (0.5,))       # 单通道灰度图，均值和标准差各一个值
])

# 下载并加载训练集
train_dataset = torchvision.datasets.MNIST(
    root='./data',          # 数据存放目录
    train=True,
    transform=transform,
    download=True           # 如果没下载过会自动下载
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

# 构建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)  # 测试集不需要打乱

class MNIST_Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.stack = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),   # 输入784 -> 隐藏128
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),    # 输出10类
        )

    def forward(self, x):
        # x 形状: (batch, 1, 28, 28)
        x = x.view(x.size(0), -1)
        # 交叉熵损失里会包含 softmax，这里直接输出 logits
        return self.stack(x)
    
model = MNIST_Classifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
# print(model)
# print(images)
# print(labels)
logits = model(images)
# print(logits.shape)
# print(pred)
# print(pred.shape)
# print(labels.shape)
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
print(loss.item())



2.318901777267456


In [47]:
x = torch.tensor([[1]])
print(x.item())

1
